# 06 - Offline DQN with Tokenizer

Same FrozenLake offline DQN loop as `02_train_offline_dqn.ipynb`, but each step is packed by `Tokenizer` and embedded by `TextEmbedder`.

- `type: "text"` — each field has its own `format=` and is tokenized as its own run. With `input_field=`, `format=` has exactly one placeholder `{field}` (a spec such as `{field:.0f}` is allowed). Omit `input_field=` for a const: keep `output_field=` and set `format=` to the literal string (no placeholder).
- `type: "token"` — integer id → one `embed_tokens` row (raw vocab id; not used here)
- `type: "image"` — vision span (VL checkpoints; not used here)
- `type: "learnable"` — scratch embedding rows with no step I/O (not used here)

There is no whole-step `format=`. Fields emit in `input_fields` order, each as a separate HF tokenize call, so BPE never merges across fields. `group_prefix=` is tokenized as `__text__` and inserted by `pack_token_batch` at the start of every `task_index` segment (and at the start of each packed sequence). Incremental decode that packs only new steps must pass `prev_grouping_ids` so a cached task does not re-emit the group prefix. Reward `0.0` and done codes `0` use `skip` / `format_skipped=","` so the value is dropped but the comma stays. `task_done` is an objective column only. `value` is a text const (`output_field="value"`, `format="\n"`, `max_tokens=1`) flagged `head_output: True`: the newline that ends each step row is the Q readout position; the tokenizer raises if that field emits more than one token.

This is a short usage example, not a full experiment. Evaluate a saved checkpoint in `09_inference.ipynb`.


In [ ]:
import torch

from mouse_core import AdamW
from mouse_core.data import (
    DataLoader,
    Augmenter,
    Tokenizer,
    compose,
    load_stores_from_hub,
)
from mouse_core.objectives import DqnObjective
from mouse_core.models import LoRAConfig, Model, Polyak, preferred_dtype, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import TextEmbedder
from mouse_core.models.heads import DiscreteActionValueHead


DATASET_ID = "mouse-example-dataset"          # Hugging Face dataset repo for load_stores_from_hub
MODEL_ID = "mouse-example-offline-dqn-text"   # Hugging Face model repo for push_model_to_hub
MAX_ACTIONS = 4                               # number of discrete actions predicted by the head
MAX_OBS_DISCRETE = 64                         # vocabulary size for discrete observations
SEQUENCE_LENGTH = 512                         # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                                # sequences per optimizer step
NUM_CYCLES = 2                               # outer train cycles (print cadence)
TRAIN_STEPS = 50                             # optimizer updates per cycle (passed to run_train)
POLYAK_TAU_HEADS = 0.0005                     # delayed Q-head interpolation (0 = frozen, 1 = copy of the online heads)
POLYAK_TAU_ENCODER = 0.0005                   # delayed encoder interpolation
POLYAK_TAU_BACKBONE = 0.0005                  # delayed backbone (LoRA adapter) interpolation


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved `Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split='train', force_download=True)

## Data pipeline

`DataLoader` samples contiguous windows up to `sequence_length` (a max) from one or more datastores. Each sequence may be shorter than the max depending on where the window starts in the store.

Pipeline order: `augmenter → tokenizer → pack → embedder`.

| Stage | Role |
| --- | --- |
| **Augmenter** | `dict → dict` (`fields=` value transforms; `seed_field=` for shared draws within a `reseed` generation). Action permute sets `input_vector_field` / `output_vector_field` on `info_q_star` so Q* stays aligned. |
| **Tokenizer** | `dict → StepTokens` (`input_field` / `output_field`; `objective_fields=` is `action` / `reward` / `episode_done` / `task_done`; `grouping_field=`) |

Compose `train_transform = compose(augmenter, tokenizer)`.
`DataLoader(transform=train_transform)` maps each step and packs into a `TokenBatch`.
Eval / decode uses the tokenizer without the augmenter so chosen actions match the env.


In [ ]:
# Pipeline order: augmenter → tokenizer

PRETRAINED = "Qwen/Qwen3-0.6B"

augmenter = Augmenter(
    seed_field="task_index",
    fields=[
        {
            "type": "discrete",
            "input_field": "action",
            "input_vector_field": "info_q_star",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "type": "discrete",
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
)

tokenizer = Tokenizer(
    input_fields=[
        {
            "type": "text",
            "input_field": "action",
            "format": "{field},",
        },
        {
            "type": "text",
            "input_field": "observation",
            "format": "{field},",
        },
        {
            "type": "text",
            "input_field": "reward",
            "format": "{field:.0f},",
            "skip": 0.0,
            "format_skipped": ",",
        },
        {
            "type": "text",
            "input_field": "episode_done",
            "format": "{field},",
            "skip": 0,
            "format_skipped": ",",
        },
        {
            "type": "text",
            "output_field": "value",
            "format": "\n",
            "max_tokens": 1,
            "head_output": True,
        },
    ],
    pretrained=PRETRAINED,
    objective_fields=[
        {
            "input_field": "action",
        },
        {
            "input_field": "reward",
        },
        {
            "input_field": "episode_done",
        },
        {
            "input_field": "task_done",
        },
    ],
    grouping_field="task_index",
    group_prefix="action,observation,reward,done\n",
)

train_transform = compose(augmenter, tokenizer)

loader = DataLoader(
    stores=stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    transform=train_transform,
    prefetch=4,
    num_workers=0,
)


## Build The Model

A Mouse Core `Model` has three main pieces:

- `TextEmbedder` looks up pretrained `embed_tokens` for the packed `__text__` ids and any `type="image"` field. Step templates and field packing live on `Tokenizer` only.
- `Qwen3Backbone` processes those tokens with a transformer backbone.
- `DiscreteActionValueHead` predicts one value per discrete action.

The backbone exposes `hidden_dim`, and the embedder and head use that same value so the pieces connect cleanly. Heads read Q from the tokenizer field flagged `head_output: True` (here `value`) via `head_output_indices`.

### What packed steps look like

Three steps (each text field is its own tokenize run; reward / episode_done skipped when zero):

```text
step 0:  action=0, observation=1, reward=0.0, episode_done=0
step 1:  action=2, observation=5, reward=0.0, episode_done=0
step 2:  action=1, observation=7, reward=1.0, episode_done=1
```

Each field fills its own `format=` (`{field},` for action / observation / episode_done, `{field:.0f},` for reward, and the const `\n` for `value`). Reward / episode_done use `format_skipped=","` so a comma stays when the value is skipped.
`group_prefix=` names the CSV columns once at the start of the `task_index` segment (here all three steps share task 0).
`*\n*` marks the head-output token (the row-ending newline) — the head reads Q from it to score the next action.

```text
task=0
action,observation,reward,done
0,1,,,*\n*
2,5,,,*\n*
1,7,1,1,*\n*
```

Packed in order: group prefix · step0 · step1 · step2. `head_output_indices` points at every `*\n*` token.


In [ ]:
backbone = Qwen3Backbone(train_kernel="flex", decode_kernel="flex", dtype=preferred_dtype(device), pretrained=PRETRAINED, lora=LoRAConfig(rank=16, alpha=32))

encoder = TextEmbedder(
    hidden_dim=backbone.hidden_dim,
    pretrained=PRETRAINED,
)

head = DiscreteActionValueHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(encoder=encoder, backbone=backbone, heads=head, action_head="action_value", reasoner=None, recurrence=None).train().to(device)
print(model)


## Training Phase

Each outer cycle runs `TRAIN_STEPS` optimizer updates via `run_train`. Mouse Core abstractions do most of the work:

1. `inputs, objective_data = loader.next_batch()` samples ragged step windows (up to `SEQUENCE_LENGTH`).
2. `model(inputs)` embeds the `TokenBatch`, runs the backbone with per-sequence causal attention/RoPE, and produces flat per-step head predictions.
3. `objective(objective_data, predictions, delayed_predictions)` computes the DQN loss and metrics.
4. `AdamW` updates weights. The backbone base is frozen bf16 and trains through its fp32 LoRA adapters (`lora=LoRAConfig(...)`); encoder and heads are fp32 too, so every update lands in fp32 with no master weights. To fine-tune the whole backbone instead, omit `lora=` and build it with `dtype=torch.float32`.
5. Delayed Q comes from the delayed model: `delayed_model = model.delayed_copy()` is a frozen copy of the online model — the fp32 encoder, LoRA adapters, and Q head are copied; the frozen bf16 base weights are shared by reference. After the online forward, `delayed_model(inputs)` runs the same `TokenBatch` through the delayed model under `torch.no_grad()`. `polyak.update(tau_heads=POLYAK_TAU_HEADS, tau_encoder=POLYAK_TAU_ENCODER, tau_backbone=POLYAK_TAU_BACKBONE)` interpolates each section toward the online model after the optimizer step: `0` keeps it frozen, `1` copies the online weights (no delay). Every interpolated parameter is fp32, so a small `tau` is never rounded away.

`DqnObjective` interprets `episode_done` and `task_done` (each `0`/`1`/`2`) through separate discount factors. The bootstrap is multiplied by the episode gamma, then by the task gamma (`1.0` when `task_done` is `0`). When a task ends both fire, so a task gamma of `0.0` zeros the whole term.


In [ ]:
optimizer = AdamW(model.parameters(), lr=1e-05, weight_decay=0.0, betas=(0.9, 0.95), eps=1e-08)
delayed_model = model.delayed_copy()
polyak = Polyak(model, delayed_model)
objective = DqnObjective(gamma_step=1.0, gamma_episode_terminal=1.0, gamma_episode_truncated=1.0, gamma_task_terminal=0.0, gamma_task_truncated=0.0, grouping_field="task_index")

def run_train(*, model: Model, delayed_model: Model, polyak: Polyak, optimizer: AdamW, objective: DqnObjective, loader: DataLoader, num_steps: int) -> tuple[torch.Tensor, dict[str, float]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    model.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        out = model(inputs)
        with torch.no_grad():
            delayed_out = delayed_model(inputs)
        loss, metrics = objective(objective_data.to(device), out.predictions, delayed_out.predictions)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        polyak.update(tau_heads=POLYAK_TAU_HEADS, tau_encoder=POLYAK_TAU_ENCODER, tau_backbone=POLYAK_TAU_BACKBONE)
    assert loss is not None
    return (loss, metrics)


## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`. Score the checkpoint later in `09_inference.ipynb`.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(model=model, delayed_model=delayed_model, polyak=polyak, optimizer=optimizer, objective=objective, loader=loader, num_steps=TRAIN_STEPS)
    print(f"cycle={cycle} train  loss={loss.item():.4f}  q={metrics['q_values_mean']:.3f}")
loader.close()


## Push To The Hub

`push_model_to_hub` saves the model architecture and weights together. Later, `load_model` can reconstruct the full `Model` without repeating the embedder, backbone, and head definitions.


In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model=model, repo_id=MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")